# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [22]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw)
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,For the Last Time,For the Last Time,$uicideboy$,9/5/2017,QM8DG1703420,"4,585",19.4,"305,049,963","65,770","5,103,054",...,NaN,2.0,"14,217",NaN,"20,104,066","13,184","50,633,006","656,337",NaN,1
4596,Dil Meri Na Sune,"Dil Meri Na Sune (From ""Genius"")",Atif Aslam,7/27/2018,INT101800122,"4,575",19.4,"52,282,360","4,602","1,449,767",...,NaN,1.0,927,NaN,NaN,NaN,NaN,"193,590",NaN,0
4597,Grace (feat. 42 Dugg),My Turn,Lil Baby,2/28/2020,USUG12000043,"4,571",19.4,"189,972,685","72,066","6,704,802",...,NaN,1.0,74,6.0,"84,426,740","28,999",NaN,"1,135,998",NaN,1
4598,Nashe Si Chadh Gayi,November Top 10 Songs,Arijit Singh,11/8/2016,INY091600067,"4,591",19.4,"145,467,020","14,037","7,387,064",...,NaN,NaN,NaN,7.0,"6,817,840",NaN,NaN,"448,292",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [18]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = np.arange(1, len(dim_tiempo) + 1)
display(dim_tiempo)

,Release Date,id_tiempo
0,4/26/2024,1
1,5/4/2024,2
2,3/19/2024,3
3,1/12/2023,4
4,5/31/2024,5
...,...,...
1557,4/9/2016,1558
1558,6/5/2023,1559
1559,10/31/2018,1560
1560,11/8/2016,1561


In [12]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Album Name', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
display(dim_pista)

,Album Name,Artist,Explicit Track
0,Million Dollar Baby - Single,Tommy Richman,0
1,Not Like Us,Kendrick Lamar,1
2,I like the way you kiss me,Artemas,0
3,Flowers - Single,Miley Cyrus,0
4,Houdini,Eminem,1
...,...,...,...
4171,ýýýýýýýýýýýý (ýýýýýýýýýýýýýý ýýýýýýýýýýýý) [pr...,BUSHIDO ZHO,1
4172,For the Last Time,$uicideboy$,1
4173,"Dil Meri Na Sune (From ""Genius"")",Atif Aslam,0
4174,November Top 10 Songs,Arijit Singh,0


In [21]:
# 3. Crear fact_streams
fact_streams = (
    spotify_raw[['Release Date', 'Spotify Streams', 'Soundcloud Streams']]
    .merge(dim_tiempo[['id_tiempo', 'Release Date']], on='Release Date', how='left')
    .drop(columns=['Release Date'])
    .drop_duplicates()
    .reset_index(drop=True)
)

display(fact_streams)

,Spotify Streams,Soundcloud Streams,id_tiempo
0,"390,470,936","4,818,457",1
1,"323,703,884","6,623,075",2
2,"601,309,283","7,208,651",3
3,"2,031,280,633",NaN,4
4,"107,034,922","207,179",5
...,...,...,...
4573,"305,049,963","50,633,006",1497
4574,"52,282,360",NaN,1030
4575,"189,972,685",NaN,746
4576,"145,467,020",NaN,1561


In [ ]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_tiempo').merge(dim_pista, on=['Album Name', 'Artist', 'Explicit Track'])

C:\Users\Acer\AppData\Local\Temp\ipykernel_37460\837010377.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in cubo_olap.select_dtypes(include='object').columns:


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track,id_tiempo,id_pista,Year,Month
0,NaN,NaN,NaN,4/26/2024,QM24S2402528,1,725.4,3.904709e+08,30716.0,196631588.0,...,18004655.0,22931.0,4818457.0,2669262.0,NaN,0,1,1,2024,4
1,NaN,NaN,NaN,5/4/2024,USUG12400910,2,545.9,3.237039e+08,28113.0,174597137.0,...,7780028.0,28444.0,6623075.0,1118279.0,NaN,1,2,2,2024,5
2,NaN,NaN,NaN,3/19/2024,QZJ842400387,3,538.4,6.013093e+08,54331.0,211607669.0,...,5022621.0,5639.0,7208651.0,5285340.0,NaN,0,3,3,2024,3
3,NaN,NaN,NaN,1/12/2023,USSM12209777,4,444.9,2.031281e+09,269802.0,136569078.0,...,190260277.0,203384.0,NaN,11822942.0,NaN,0,4,4,2023,1
4,NaN,NaN,NaN,5/31/2024,USUG12403398,5,423.3,1.070349e+08,7223.0,151469874.0,...,4493884.0,7006.0,207179.0,457017.0,NaN,1,5,5,2024,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,NaN,NaN,NaN,9/5/2017,QM8DG1703420,4585,19.4,3.050500e+08,65770.0,5103054.0,...,20104066.0,13184.0,50633006.0,656337.0,NaN,1,1497,4586,2017,9
4596,NaN,NaN,NaN,7/27/2018,INT101800122,4575,19.4,5.228236e+07,4602.0,1449767.0,...,NaN,NaN,NaN,193590.0,NaN,0,1030,4587,2018,7
4597,NaN,NaN,NaN,2/28/2020,USUG12000043,4571,19.4,1.899727e+08,72066.0,6704802.0,...,84426740.0,28999.0,NaN,1135998.0,NaN,1,746,4588,2020,2
4598,NaN,NaN,NaN,11/8/2016,INY091600067,4591,19.4,1.454670e+08,14037.0,7387064.0,...,6817840.0,NaN,NaN,448292.0,NaN,0,1561,4589,2016,11


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [ ]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Escribe aquí si es Roll-up, Drill-down, Slice o Dice]
respuesta_1 = # Tu código...

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_2 = # Tu código...

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: [Escribe aquí...]
respuesta_3 = # Tu código...

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_4 = # Tu código...

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: [Escribe aquí...]
respuesta_5 = # Tu código...

In [ ]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Escribe aquí...]
respuesta_6 = # Tu código...

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: [Escribe aquí...]
respuesta_7 = # Tu código...

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: [Escribe aquí...]
respuesta_8 = # Tu código...

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: [Escribe aquí...]
respuesta_9 = # Tu código...

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_10 = # Tu código...